# Bi-level pruning - CIFAR-10 (VGG16-BN / ResNet-56)

Doi chieu voi so **published** cua L1 / HRank / GAL / SSS / CORING / SPSRC ma khong
phai chay lai tung baseline. Protocol lay tu repo CORING (github.com/vantienpham/CORING).

| Truc fair | Cach xu ly |
|---|---|
| Transform | RandomCrop(32,pad=4) + HFlip, std **(.2023,.1994,.2010)** - dung `main/data/cifar10.py` |
| Kien truc | ResNet-56 shortcut **option A** (zero-pad) = 0.853M, dung nhu HRank |
| Diem xuat phat | Nap thang `resnet_56.pt` cua HRank - **cung checkpoint voi CORING** |
| Recipe finetune | `coring`: 300 ep, lr 0.01, x0.1@150,225, wd 5e-3, bs 128 |
| Ngan sach epoch | Vong prune tieu bao nhieu thi finetune tru bay nhieu -> tong = 300 |
| Muc nen | `--match-macs` tu tim target-sparsity de rot dung diem cua CORING |
| Loai method | Chay ca `structured-only` (cung loai CORING) va `bi-level` (ablation) |

**Thu tu:** Cell 1-4 setup -> Cell 5 SMOKE -> Cell 6 dense (bo qua neu co pretrained) -> Cell 7 chay that.

**Truoc khi chay:** Settings > Accelerator > **GPU**, Settings > **Internet ON**.


In [ ]:
# --- 1. Clone repo + checkout branch dev (repo public -> khong can token) ---
import os, subprocess, sys, json, time

REPO   = '/kaggle/working/OnestageDetectionPunner'
URL    = 'https://github.com/barone04/OnestageDetectionPunner.git'
BRANCH = 'dev'

if not os.path.isdir(os.path.join(REPO, '.git')):
    r = subprocess.run(f'git clone -q --branch {BRANCH} {URL} {REPO}', shell=True)
    assert r.returncode == 0, ('Clone that bai. Kiem tra Settings > Internet ON. '
                               'Neu repo doi sang private thi dung '
                               'https://<TOKEN>@github.com/... trong URL.')
    print('Da clone.')
else:
    print('Repo da co, chi pull.')

os.chdir(REPO)
subprocess.run(f'git checkout -q {BRANCH} && git pull -q origin {BRANCH}', shell=True)
subprocess.run('git log --oneline -1', shell=True)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('cwd:', os.getcwd())


In [ ]:
# --- 2. WANDB_API_KEY: uu tien Kaggle Secrets, roi moi den file .env ---
# Cach 1 (nen dung): Add-ons > Secrets > Add secret, Label = WANDB_API_KEY.
#   Khong can tao Dataset, khong de key nam trong file.
# Cach 2: Dataset chua file .env (dat duong dan vao ENV_PATH ben duoi).
ENV_PATH = '/kaggle/input/datasets/bophaninhthi/env-file1/.env'

def _from_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('WANDB_API_KEY')
    except Exception as e:
        print('  Secrets:', type(e).__name__, str(e)[:80])
        return None

def _from_env_file(path):
    if not os.path.isfile(path):
        print('  .env: khong thay', path)
        return None
    for line in open(path):
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip().strip(chr(34)).strip(chr(39))
    return os.environ.get('WANDB_API_KEY')

key = _from_secrets() or _from_env_file(ENV_PATH)
if key:
    os.environ['WANDB_API_KEY'] = key

USE_WANDB = bool(os.environ.get('WANDB_API_KEY'))
print('wandb:', 'ON' if USE_WANDB else 'OFF (van train binh thuong, chi khong log)')


In [ ]:
# --- 3. Deps + kiem tra moi truong ---
subprocess.run('pip -q install thop' + (' wandb' if USE_WANDB else ''), shell=True)
import torch, torchvision, thop
print('torch', torch.__version__, '| torchvision', torchvision.__version__)
print('cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Bat GPU: Settings > Accelerator > GPU'


In [ ]:
# --- 4. Config ---
MODEL    = 'resnet56'    # 'resnet56' hoac 'vgg16'
VGG_HEAD = 'hrank'       # chi cho vgg16. hrank=14.99M (HRank/CORING) | single=14.72M (SPSRC)
PROTOCOL = 'coring'      # recipe finetune: 'coring' (300ep) | 'spsrc' (80ep)
SEED     = 0

# Moi model mot wandb project rieng, dat ten theo dung quy uoc san co cua repo:
# resnet56-cifar10-norton / -sliming / -coring  ->  resnet56-cifar10-bilevel
PROJECT  = f'{MODEL}-cifar10-bilevel'
os.environ['WANDB_PROJECT'] = PROJECT

# Checkpoint dense cua HRank -> CUNG diem xuat phat voi CORING (ho khong tu train).
#   wget https://github.com/vantienpham/CORING/releases/download/v0.1.0/resnet_56.pt
#   wget https://github.com/vantienpham/CORING/releases/download/v0.1.0/vgg_16_bn.pt
# Upload len Kaggle nhu 1 Dataset (ten gi cung duoc - o duoi tu do theo TEN FILE).
# Dat None neu muon tu train dense o Cell 6 (~1.5h, kem fair hon).
CKPT_NAME  = 'resnet_56.pt' if MODEL == 'resnet56' else 'vgg_16_bn.pt'
PRETRAINED = f'/kaggle/input/hrank-ckpt/{CKPT_NAME}'

if PRETRAINED and not os.path.isfile(PRETRAINED):
    import glob
    hit = glob.glob(f'/kaggle/input/**/{CKPT_NAME}', recursive=True)
    if hit:
        PRETRAINED = hit[0]
        print('Tu tim thay checkpoint:', PRETRAINED)
    else:
        print(f'!! KHONG thay {CKPT_NAME} o bat ky dau trong /kaggle/input.')
        print('   Dang co nhung thu muc nay:')
        for d in sorted(glob.glob('/kaggle/input/*')):
            print('    -', d, sorted(os.listdir(d))[:6])
        print('   -> Bam "+ Add Input" > Datasets, chon dataset chua ' + CKPT_NAME + ',')
        print('      hoac dat PRETRAINED = None de tu train dense o Cell 6.')

# So PUBLISHED cua CORING (README repo cua ho) - dung lam moc va lam diem can match.
#   ResNet-56 baseline : 93.26%  | 0.85M params | 125.49M FLOPs
#   CORING-E-5         : 94.76%  | 0.66M (-22.4%) |  91.23M (-27.3%)
#   CORING-E           : 92.84%  | 0.24M (-71.8%) |  34.79M (-72.3%)
CORING_REF = {'baseline': dict(top1=93.26, params=0.85, macs=125.49),
              'E-5':      dict(top1=94.76, params=0.66, macs=91.23),
              'E':        dict(top1=92.84, params=0.24, macs=34.79)}

TARGETS = [91.23]         # diem nen can match (MACs trieu). CORING-E-5=91.23, CORING-E=34.79
MATCH_METRIC = 'macs'     # 'macs' | 'params'

# ABLATION de xuat #2 va #3. Tat ca BI-LEVEL: #3 vo nghia khi khong co tang
# unstructured (moi filter mat do 100% -> tu roi ve norm).
#   (A) cat BAO NHIEU moi lop : --layer-schedule uniform | sensitivity (#2)
#   (B) cap nao DU THUA       : khoang cach L1-inf-inf — KHONG doi o dong nao
#   (C) giet CAI NAO trong cap: --kill-by norm | density (#3)
# Dong baseline (uniform + norm) DA CO: resnet56_macs91.23_bi-level_ep400 = 93.82%.
ABLATION = {
    'kill-density':       '--layer-schedule uniform --kill-by density',       # chi #3
    'alloc-sens':         '--layer-schedule sensitivity --kill-by norm',      # chi #2
    'alloc-sens_kill-density': '--layer-schedule sensitivity --kill-by density',  # #2+#3
}
ALPHA = 0.5               # #2: 0 = uniform, 1 = theo sat do nhay

# Tong ngan sach epoch (vong prune + finetune). Cac method trong Table 3 dung rat khac:
#   HRank 30 | TPP 120 | CORING 300 | CHIP 400
# GIU 400 de so duoc voi dong baseline 93.82% (cung 400 ep).
BUDGET = 400

DATA = '/kaggle/working/data'
OUT  = '/kaggle/working/output/cifar'
DENSE_DIR = f'{OUT}/{MODEL}_dense'
os.makedirs(DATA, exist_ok=True)

WB = '--wandb' if USE_WANDB else ''
# SRC dat ngay o day: co pretrained thi Cell 7 chay duoc ke ca khi bo qua Cell 6.
SRC = f'--pretrained {PRETRAINED}' if PRETRAINED else None
head = f'--vgg-head {VGG_HEAD}' if MODEL == 'vgg16' else ''

def run(cmd):
    """Chay lenh, in truc tiep, dung han neu that bai."""
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise SystemExit(f'That bai (exit {r.returncode}): {cmd}')

n_run = len(TARGETS) * len(ABLATION)
print(f'{MODEL} | protocol={PROTOCOL} | budget={BUDGET} ep | alpha={ALPHA} | targets={TARGETS} {MATCH_METRIC}')
print(f'wandb project: {PROJECT} | so run se tao: {n_run} ({len(TARGETS)} rate x {len(ABLATION)} dong ablation)')
for name in ABLATION:
    print(f'   {MODEL}_{MATCH_METRIC}{TARGETS[0]}_bi-level_ep{BUDGET}_{name}')
print('pretrained:', PRETRAINED or '(tu train dense)')


## 5. Smoke test

Kiem tra truoc khi dot GPU:
1. Model dung so params published (ResNet-56 A = 0.853M, VGG16-BN hrank = 14.99M)
2. Surgery cho lean model **giong het** masked model (`max|diff| < 1e-7`)
3. Checkpoint pretrained nap duoc, khong thieu/thua key
4. Ca 3 mode chay thong voi 1 epoch

~5 phut. Fail o day thi dung.


In [ ]:
# --- 5a. Self-check model + surgery ---
run('python -m models.cifar')
run('python -m pruning.surgery_cifar')


In [ ]:
# --- 5b. Checkpoint pretrained: nap duoc khong, va co dung model khong ---
if PRETRAINED:
    assert os.path.isfile(PRETRAINED), PRETRAINED
    from models.cifar import build_cifar_model, load_hrank_state_dict
    import cifar as C

    _m = build_cifar_model(C.build(MODEL, 10, 'all', VGG_HEAD).config())
    load_hrank_state_dict(_m, PRETRAINED)          # raise neu lech key
    _p, _mac = C.count_cost(_m, torch.device('cpu'))
    print(f'Dense: {_p:.3f}M params | {_mac:.2f}M MACs')

    if MODEL == 'resnet56':
        # Doi chieu voi so published cua CORING -> bat sai kien truc NGAY,
        # truoc khi dot vai gio GPU vao mot model khac cua ho.
        ref = CORING_REF['baseline']
        assert abs(_p - ref['params']) < 0.02, f"params {_p:.3f}M != {ref['params']}M"
        assert abs(_mac - ref['macs']) / ref['macs'] < 0.02, \
            f"MACs {_mac:.2f}M != {ref['macs']}M -> kien truc khong khop CORING"
        print(f"OK khop published: {ref['params']}M params, {ref['macs']}M FLOPs, "
              f"top1 {ref['top1']}%")
else:
    print('Khong co pretrained -> se train dense o Cell 6')


In [ ]:
# --- 5c. Smoke: dense 1 epoch (--skip-gate vi 1 epoch chac chan khong dat moc) ---
SMOKE = f'{OUT}/_smoke'
run(f'python cifar.py --mode dense --model {MODEL} {head} --data-path {DATA} '
    f'--epochs 1 --skip-gate --output-dir {SMOKE}_dense')


In [ ]:
# --- 5d. Smoke: prune (2 vong x 1 ep) + calibration do nhay + kill-by density ---
# Chay DUNG duong code moi (#2 + #3) truoc khi dot GPU vao run that.
src = f'--pretrained {PRETRAINED}' if PRETRAINED else f'--checkpoint {SMOKE}_dense/model_best.pth'
run(f'python cifar.py --mode prune --model {MODEL} {head} --data-path {DATA} '
    f'{src} --protocol {PROTOCOL} --match-{MATCH_METRIC} {TARGETS[0]} '
    f'--layer-schedule sensitivity --alpha {ALPHA} --kill-by density --calib-images 256 '
    f'--prune-iters 2 --prune-finetune-epochs 1 --output-dir {SMOKE}_p')
cost = json.load(open(f'{SMOKE}_p/cost.json'))
sens = json.load(open(f'{SMOKE}_p/sensitivity.json'))
print(cost)
assert cost['kill_by'] == 'density' and cost['layer_schedule'] == 'sensitivity', cost
assert len(sens['kl']) == len(sens['weights']) > 0, sens
print(f"sensitivity.json: {len(sens['kl'])} lop, trong so cat "
      f"{min(sens['weights']):.3f}..{max(sens['weights']):.3f}")

In [ ]:
# --- 5e. Smoke: finetune 1 epoch tren lean model ---
run(f'python cifar.py --mode finetune --data-path {DATA} --protocol {PROTOCOL} '
    f'--lean {SMOKE}_p/model_lean.pth --epochs 1 --output-dir {SMOKE}_ft')
print(chr(10) + 'SMOKE TEST PASS - duoc phep chay that.')


## 6. Dense baseline

Co `PRETRAINED` thi cell nay **khong train gi**, chay het ~0 giay - `SRC` da duoc
dat o Cell 4 roi. Van nen bam chay de thay xac nhan, nhung bo qua cung khong sao.

Dung checkpoint HRank fair hon han vi CORING cung xuat phat tu dung file do,
chu khong tu train.

Neu KHONG co pretrained: ResNet-56 200 ep ~1-1.5h, va cell tu **dung** neu top-1
lech >0.5% so voi published (ResNet-56 93.26% | VGG16-BN 93.96%) - prune tren dense
sai moc thi khong duoc trich bang cua ho.


In [ ]:
# --- 6. Dense: chi TRAIN khi khong co pretrained (co pretrained thi cell nay ~0s) ---
if PRETRAINED:
    print('Co pretrained -> khong train dense. SRC =', SRC)
else:
    t0 = time.time()
    run(f'python cifar.py --mode dense --model {MODEL} {head} --data-path {DATA} '
        f'--seed {SEED} --output-dir {DENSE_DIR} {WB}')
    print(f'Dense xong sau {(time.time()-t0)/3600:.2f}h')
    SRC = f'--checkpoint {DENSE_DIR}/model_best.pth'
    print('SRC =', SRC)


## 7. Ablation de xuat #2 va #3

Ba quyet dinh trong moi lop, moi de xuat cham dung mot cho:

| | (A) cat bao nhieu | (B) cap nao du thua | (C) giet cai nao |
|---|---|---|---|
| baseline (**da co: 93.82%**) | uniform | L1-inf-inf | norm |
| `kill-density` (#3) | uniform | L1-inf-inf | **mat do** |
| `alloc-sens` (#2) | **do nhay** | L1-inf-inf | norm |
| `alloc-sens_kill-density` | **do nhay** | L1-inf-inf | **mat do** |

Cot (B) giong nhau o moi dong — khoang cach L1-inf-inf khong doi.

- **#3 `--kill-by density`**: trong cap du thua, giet filter co it trong so khac 0 hon sau
  tang unstructured. Do tren 200 filter: mat do tuong quan voi tam quan trong that rho=0.325,
  norm chi rho=0.035.
- **#2 `--layer-schedule sensitivity`**: truoc khi cat, do tung lop chiu duoc thao tac zero-hoa
  cua tang 1 den dau (KL tren 2000 anh TRAIN), roi phan bo ti le cat: lop it nhay cat nhieu.
  Tong muc nen van do `--match-macs` quyet dinh. Do nhay bien thien TRONG stage gap 5.4x GIUA
  stage — ly do `chip`/`steep` (chinh o cap stage) that bai.

Tat ca **bi-level**, 400 ep, diem 91.23M. 3 run x ~2h = ~6h.

**wandb** (project `resnet56-cifar10-bilevel`): metric moi epoch nhu cu
(`train/loss` `train/acc` `val/loss` `val/acc` `lr` `stage`), summary them `kill_by`,
`alpha`, `layer_schedule`, `best_top1`. Run co #2 them bang `sensitivity/per_layer`
(KL, trong so cat, ti le cat tung lop).

In [ ]:
# --- 7. Chay that: moi target x moi dong ablation (bi-level) ---
results = {}
for tgt in TARGETS:
    for name, flags in ABLATION.items():
        tag = f'{MODEL}_{MATCH_METRIC}{tgt}_bi-level_ep{BUDGET}_{name}'
        p_dir, f_dir = f'{OUT}/{tag}', f'{OUT}/{tag}_ft'
        # cung --wandb-run cho ca 2 buoc -> 1 wandb run duy nhat cho cau hinh nay
        wb = f'{WB} --wandb-run {tag}' if WB else ''
        print(chr(10) + '=' * 64 + chr(10) + f' {tag}' + chr(10) + '=' * 64, flush=True)

        run(f'python cifar.py --mode prune --model {MODEL} {head} --data-path {DATA} '
            f'{SRC} --protocol {PROTOCOL} --seed {SEED} {wb} --budget {BUDGET} '
            f'{flags} --alpha {ALPHA} --match-{MATCH_METRIC} {tgt} '
            f'--prune-iters 5 --prune-finetune-epochs 3 --output-dir {p_dir}')

        run(f'python cifar.py --mode finetune --data-path {DATA} --protocol {PROTOCOL} '
            f'--seed {SEED} {wb} --budget {BUDGET} '
            f'--lean {p_dir}/model_lean.pth --output-dir {f_dir}')

        cost = json.load(open(f'{p_dir}/cost.json'))
        ck = torch.load(f'{f_dir}/model_best.pth', map_location='cpu', weights_only=False)
        results[tag] = dict(cost, top1=ck['top1'], ablation=name)
        print(f"{tag} -> best top1={ck['top1']:.2f}%  {cost}")

In [ ]:
# --- 8. Bang ablation (dong baseline lay tu run da chay) ---
BASELINE = dict(ablation='baseline (uniform+norm)', layer_schedule='uniform', kill_by='norm',
                top1=93.82, params_M=0.615292, macs_M=91.644544)     # _ep400, da chay
rows = [BASELINE] + list(results.values())
cols = ('dong', '(A) bao nhieu', '(C) giet cai nao', 'best top1', 'params M', 'MACs M', 'vs base')
print('{:<26} | {:>13} | {:>16} | {:>9} | {:>8} | {:>8} | {:>7}'.format(*cols))
print('-' * 104)
for r in rows:
    print('{:<26} | {:>13} | {:>16} | {:>8.2f}% | {:>8.3f} | {:>8.2f} | {:>+7.2f}'.format(
        r['ablation'], r['layer_schedule'], r['kill_by'], r['top1'],
        r['params_M'], r['macs_M'], r['top1'] - BASELINE['top1']))
print(chr(10) + 'Nhac lai: cung cau hinh da dao dong +-0.3 giua cac lan chay.')
print('  > +0.5 -> nhieu kha nang that | < 0.3 -> can them seed.')
path = f'{OUT}/summary_{MODEL}_{PROTOCOL}_ablation.json'
json.dump(results, open(path, 'w'), indent=2)
print(chr(10) + 'Luu: ' + path)